# Crezee 2022 peatland and nearby open-water outline

This Colab notebook mounts Google Drive and combines two Crezee 2022 rasters. Peatland is defined as valid peat-thickness pixels **greater than 0.01 m**; open water is **land-cover class 1**. Both masks are aggregated to a shared **500 m** equal-area grid, and open-water cells within 10 km of peatland are included. Disconnected polygon components smaller than **100 km²** are removed, the remainder is dissolved to one feature, and the result is exported in **EPSG:4326** for upload as a Google Earth Engine table asset.

## 1. Install and import the geospatial packages

In [ ]:
!pip -q install rasterio geopandas matplotlib scipy

In [ ]:
from pathlib import Path
import shutil

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from google.colab import drive, files
from rasterio.enums import Resampling
from rasterio.features import shapes
from rasterio.warp import calculate_default_transform, reproject
from scipy.ndimage import distance_transform_edt
from shapely.geometry import shape
from shapely.ops import unary_union

## 2. Mount Drive and find both GeoTIFFs

The path below matches `My Drive > Colab Notebooks > PanAfrica_LU`. The first pattern finds the peat-thickness raster; the second finds the `Crezee_2022` land-cover classification raster containing `Classification` and `Most_likely_class` in its filename.

In [ ]:
drive.mount('/content/drive')

data_dir = Path('/content/drive/MyDrive/Colab Notebooks/PanAfrica_LU')
peat_pattern = 'Crezee_2022_Median_Peat_thickness_RF_100runs*.tif'
landcover_pattern = 'Crezee_2022_*Classification*Most_likely_class*.tif'
peat_matches = sorted(data_dir.glob(peat_pattern))
landcover_matches = sorted(data_dir.glob(landcover_pattern))

if not peat_matches:
    raise FileNotFoundError(
        f'No {peat_pattern} found in {data_dir}. '
        'Check the folder and filename in Google Drive.'
    )
if not landcover_matches:
    raise FileNotFoundError(
        f'No {landcover_pattern} found in {data_dir}. '
        'Check the folder and filename in Google Drive.'
    )

tif_path = peat_matches[0]
landcover_path = landcover_matches[0]
print(f'Peat thickness: {tif_path}')
print(f'Land cover:     {landcover_path}')

## 3. Inspect and display the map

Only a downsampled display array is loaded for the map. The peatland mask is created from the original raster in blocks, avoiding a full-resolution array in RAM.

In [ ]:
max_display_side = 2000

with rasterio.open(tif_path) as src:
    if src.crs is None:
        raise ValueError('The GeoTIFF has no CRS; a georeferenced shapefile cannot be created safely.')

    scale = min(1.0, max_display_side / max(src.width, src.height))
    display_width = max(1, round(src.width * scale))
    display_height = max(1, round(src.height * scale))
    raster = src.read(
        1,
        out_shape=(display_height, display_width),
        resampling=Resampling.bilinear,
        masked=True,
    )
    bounds = src.bounds
    raster_crs = src.crs
    raster_shape = (src.height, src.width)
    nodata = src.nodata

print(f'Raster size: {raster_shape[1]:,} x {raster_shape[0]:,} pixels')
print(f'CRS: {raster_crs}')
print(f'Bounds: {bounds}')
print(f'NoData: {nodata}')

fig, ax = plt.subplots(figsize=(14, 10))
image = ax.imshow(
    raster,
    extent=(bounds.left, bounds.right, bounds.bottom, bounds.top),
    origin='upper',
    interpolation='nearest',
    cmap='viridis',
)
ax.set_title(tif_path.name)
ax.set_xlabel('Easting / longitude')
ax.set_ylabel('Northing / latitude')
ax.set_aspect('equal')
fig.colorbar(image, ax=ax, shrink=0.75, label='Peat thickness (m)')
plt.show()

## 4. Create and filter the 500 m peatland and nearby-water outline

A valid peat-thickness pixel is peatland when it exceeds 0.01 m, and land-cover class 1 is open water. Both binary masks are aggregated with `max` to the same 500 m EPSG:6933 grid, so a grid cell is flagged when it contains any matching source pixel. Open-water cells are retained only when within 10 km of a peatland cell. After polygonization, disconnected components smaller than 100 km² are removed using equal-area geometry measurements.

In [ ]:
peat_threshold_m = 0.01
water_class = 1
grid_size_m = 500
water_proximity_m = 10_000
minimum_component_km2 = 100
target_crs = 'EPSG:6933'  # Equal-area projection with metre units
peat_mask_path = Path('/content/peatland_mask.tif')
water_mask_path = Path('/content/open_water_mask.tif')
print(f'Peatland threshold: > {peat_threshold_m} m')
print(f'Open-water class: {water_class}')

# Write a compact 0/1 mask block by block, avoiding a full-resolution array in RAM.
with rasterio.open(tif_path) as src:
    mask_profile = src.profile.copy()
    mask_profile.update(count=1, dtype='uint8', nodata=0, compress='deflate')
    with rasterio.open(peat_mask_path, 'w', **mask_profile) as mask_dst:
        for _, window in src.block_windows(1):
            block = src.read(1, window=window, masked=True)
            valid = (~np.ma.getmaskarray(block)) & np.isfinite(block.data)
            peatland = valid & (block.data > peat_threshold_m)
            mask_dst.write(peatland.astype('uint8'), 1, window=window)


# Create the open-water mask from land-cover class 1.
with rasterio.open(landcover_path) as src:
    if src.crs is None:
        raise ValueError('The land-cover GeoTIFF has no CRS.')
    water_profile = src.profile.copy()
    water_profile.update(count=1, dtype='uint8', nodata=0, compress='deflate')
    with rasterio.open(water_mask_path, 'w', **water_profile) as mask_dst:
        for _, window in src.block_windows(1):
            block = src.read(1, window=window, masked=True)
            valid = ~np.ma.getmaskarray(block)
            open_water = valid & (block.data == water_class)
            mask_dst.write(open_water.astype('uint8'), 1, window=window)

# Define a shared 500 m grid from the peat raster extent.
with rasterio.open(peat_mask_path) as mask_src:
    coarse_transform, coarse_width, coarse_height = calculate_default_transform(
        mask_src.crs, target_crs, mask_src.width, mask_src.height,
        *mask_src.bounds, resolution=grid_size_m,
    )
    peat_grid = np.zeros((coarse_height, coarse_width), dtype='uint8')
    reproject(
        source=rasterio.band(mask_src, 1),
        destination=peat_grid,
        src_transform=mask_src.transform,
        src_crs=mask_src.crs,
        dst_transform=coarse_transform,
        dst_crs=target_crs,
        src_nodata=0,
        dst_nodata=0,
        resampling=Resampling.max,
    )

# Aggregate open water onto exactly the same grid.
water_grid = np.zeros((coarse_height, coarse_width), dtype='uint8')
with rasterio.open(water_mask_path) as water_src:
    reproject(
        source=rasterio.band(water_src, 1),
        destination=water_grid,
        src_transform=water_src.transform,
        src_crs=water_src.crs,
        dst_transform=coarse_transform,
        dst_crs=target_crs,
        src_nodata=0,
        dst_nodata=0,
        resampling=Resampling.max,
    )

peat_grid = peat_grid.astype(bool)
water_grid = water_grid.astype(bool)
if not peat_grid.any():
    raise ValueError(f'No valid pixels exceed {peat_threshold_m} m peat thickness.')
distance_to_peat_m = distance_transform_edt(~peat_grid, sampling=grid_size_m)
nearby_water = water_grid & (distance_to_peat_m <= water_proximity_m)
combined_mask = peat_grid | nearby_water
selected_cell_count = int(combined_mask.sum())
print(f'Coarse grid: {coarse_width:,} × {coarse_height:,} cells')
print(f'Peatland 500 m cells: {int(peat_grid.sum()):,}')
print(f'Nearby open-water cells added: {int(nearby_water.sum()):,}')
print(f'Combined cells: {selected_cell_count:,}')

polygons = [
    shape(geometry)
    for geometry, value in shapes(
        combined_mask.astype('uint8'), mask=combined_mask,
        transform=coarse_transform, connectivity=8,
    )
    if value == 1
]
component_areas_km2 = np.array([polygon.area / 1_000_000 for polygon in polygons])
retained_polygons = [
    polygon for polygon, area_km2 in zip(polygons, component_areas_km2)
    if area_km2 >= minimum_component_km2
]
print(f'Connected components before filtering: {len(polygons):,}')
print(f'Components retained (>= {minimum_component_km2} km²): {len(retained_polygons):,}')
if not retained_polygons:
    raise ValueError(
        f'No connected polygon component is at least {minimum_component_km2} km².'
    )
selected_geometry = unary_union(retained_polygons)
outline = gpd.GeoDataFrame(
    {
        'name': ['peat_water'],
        'minpeat_m': [peat_threshold_m],
        'grid_m': [grid_size_m],
        'water_km': [water_proximity_m / 1000],
        'min_km2': [minimum_component_km2],
    },
    geometry=[selected_geometry],
    crs=target_crs,
)

ax = outline.plot(figsize=(12, 8), facecolor='none', edgecolor='red', linewidth=0.5)
ax.set_title('Peatland + nearby water at 500 m; components >= 100 km²')
ax.set_xlabel('Easting / longitude')
ax.set_ylabel('Northing / latitude')
ax.set_aspect('equal')
plt.show()

outline

## 5. Reproject, export, and download the shapefile

The 500 m grid and 100 km² component filter were calculated in EPSG:6933. The finished outline is now reprojected to EPSG:4326 (longitude/latitude), a widely supported CRS for Google Earth Engine table-asset uploads. A shapefile consists of several companion files, so all components—including the `.prj` CRS definition—are bundled into one ZIP.

In [ ]:
output_dir = Path('/content/Crezee_2022_peatland_nearby_water')
output_dir.mkdir(parents=True, exist_ok=True)
shapefile_path = output_dir / 'Crezee_2022_peatland_nearby_water.shp'

gee_outline = outline.to_crs('EPSG:4326')
if gee_outline.crs.to_epsg() != 4326:
    raise ValueError(f'Unexpected export CRS: {gee_outline.crs}')
gee_outline.to_file(shapefile_path, driver='ESRI Shapefile', index=False, encoding='UTF-8')
zip_path = Path(shutil.make_archive('/content/Crezee_2022_peatland_nearby_water', 'zip', output_dir))

print(f'Export CRS: {gee_outline.crs}')
print(f'Created: {zip_path}')
files.download(str(zip_path))